<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Synthetic Financial Time Series

**Spectral-Normalised GAN/WGAN-GP (PyTorch)**

&copy; Dr. Yves J. Hilpisch

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>

Install the package before running this notebook. Set `SYNTHTS_DATA` to
a frozen local CSV to avoid a live download. Sequential/TimeGAN examples
accept an explicitly selected YAML via `SYNTHTS_CONFIG`; historical
selected settings and their recorded metrics are not loaded automatically.

## Imports & Settings

In [ ]:
import os
import numpy as np
import pandas as pd
from pylab import plt, mpl

In [ ]:
import torch

In [ ]:
from synthts.data import (
    EOD_URL, load_eod, compute_log_returns, ReturnScaler, split_returns,
)
from synthts.wgangp import (WGANGenerator, Critic,
    train_wgangp, generate_samples)
from synthts.evaluate import (ks_test, ks_test_batch, pass_rate,
    describe_comparison, mse_sorted,
    plot_histograms, plot_sorted,
    plot_cumulative, plot_pvalue_histogram)

In [ ]:
plt.style.use('seaborn-v0_8')
#mpl.rcParams['figure.dpi'] = 300
#mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['font.family'] = 'serif'

## Simple Example — Cubic Function

The examples use `n_critic=1`: spectral normalisation with binary
cross-entropy. Set `n_critic>1` for Wasserstein loss with gradient penalty.

In [ ]:
x = np.linspace(-2, 2, 500)
y = x ** 3
scaler = ReturnScaler()
y_ = scaler.fit_transform(y)

In [ ]:
torch.manual_seed(100)
g = WGANGenerator(latent_dim=2, hidden_units=32, output_dim=1)
c = Critic(input_dim=1, hidden_units=32)

In [ ]:
%%time
history = train_wgangp(
    g, c, y_.reshape(-1, 1),
    epochs=10001, batch_size=64,
    n_critic=1, lr_g=0.0001, lr_c=0.001,
    latent_dim=2, seed=100
)

### Evaluation

In [ ]:
data = pd.DataFrame({'real': y}, index=x)
N = 5
for i in range(N):
    samples = generate_samples(g, len(y), latent_dim=2)
    data[f'synth_{i:02d}'] = scaler.inverse_transform(samples)

In [ ]:
data.describe().round(3)

In [ ]:
mse_sorted(data['real'].values, data['synth_00'].values)

In [ ]:
data.apply(np.sort).plot(
    style=['r'] + N * ['b--'], lw=1, legend=False);

## Financial Example — GLD Log-Returns

In [ ]:
prices = load_eod(url=os.environ.get('SYNTHTS_DATA', EOD_URL),
                  symbol='GLD', n_obs=2 * 252)
rets = compute_log_returns(prices)
scaler = ReturnScaler()
rets_ = scaler.fit_transform(rets)

In [ ]:
torch.manual_seed(100)
g = WGANGenerator(latent_dim=2, hidden_units=32, output_dim=1)
c = Critic(input_dim=1, hidden_units=32)

In [ ]:
%%time
history = train_wgangp(
    g, c, rets_.reshape(-1, 1),
    epochs=10001, batch_size=64,
    n_critic=1, lr_g=0.0001, lr_c=0.001,
    latent_dim=2, seed=100
)

### Generating Synthetic Returns

In [ ]:
data = pd.DataFrame({'real': rets})
N = 25
for i in range(N):
    samples = generate_samples(g, len(rets_), latent_dim=2)
    data[f'synth_{i:02d}'] = scaler.inverse_transform(samples)

In [ ]:
describe_comparison(data, n_show=5)

In [ ]:
plot_histograms(data['real'].values, data['synth_00'].values);

In [ ]:
plot_sorted(data['real'].values, data['synth_00'].values);

In [ ]:
plot_cumulative(data, n_synth=N);

### Kolmogorov-Smirnov Test

Non-rejection is a descriptive diagnostic, not evidence of equivalence.
Serial dependence and overlapping windows prevent interpreting these
p-values as calibrated independent-sample tests.

In [ ]:
synth_cols = [c for c in data.columns if c.startswith('synth')]
pvals = ks_test_batch(data['real'].values, data[synth_cols])
pvals.round(4)

In [ ]:
f'KS pass rate: {pass_rate(pvals):.0%}'

In [ ]:
plot_pvalue_histogram(pvals);

<img src="https://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="https://tpq.io" target="_blank">https://tpq.io</a> | <a href="https://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a>